# Simple Fuzzy Battery Management — EV Charging Controller
**Milestone-3: Phase 4 (Testing) · Phase 5 (Analysis) · Phase 6 (User Interface)**

This notebook builds on the Mamdani fuzzy inference system for adaptive EV charging-current control
(5 → actually 4 crisp inputs implemented: **SOC, Battery Temperature, Cell Voltage Difference, Ambient Temperature**),
runs it against 10 different input combinations, discusses the controller's behaviour, and wraps it in an
interactive user interface.

## Setup — Fuzzy Inference System
Same system as the original controller, refactored into a reusable function `get_charging_current()` so it can be called repeatedly for testing and for the UI (no more blocking `input()` calls).

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# ---------- Antecedents (inputs) ----------
soc = ctrl.Antecedent(np.arange(0, 101, 1), 'soc')
bt  = ctrl.Antecedent(np.arange(0, 61, 1), 'bt')
dv  = ctrl.Antecedent(np.arange(0, 101, 1), 'dv')
at  = ctrl.Antecedent(np.arange(15, 41, 1), 'at')

# ---------- Consequent (output) ----------
current = ctrl.Consequent(np.arange(2, 20.1, 0.1), 'current')

# ---------- Membership functions ----------
soc['low']    = fuzz.trimf(soc.universe, [0, 0, 40])
soc['medium'] = fuzz.trimf(soc.universe, [20, 50, 80])
soc['high']   = fuzz.trimf(soc.universe, [60, 100, 100])

bt['normal'] = fuzz.trimf(bt.universe, [15, 25, 35])
bt['warm']   = fuzz.trimf(bt.universe, [30, 40, 50])
bt['hot']    = fuzz.trimf(bt.universe, [45, 60, 60])

dv['small']  = fuzz.trimf(dv.universe, [0, 0, 30])
dv['medium'] = fuzz.trimf(dv.universe, [20, 50, 80])
dv['large']  = fuzz.trimf(dv.universe, [60, 100, 100])

at['cool']   = fuzz.trimf(at.universe, [15, 15, 25])
at['normal'] = fuzz.trimf(at.universe, [20, 27, 34])
at['hot']    = fuzz.trimf(at.universe, [30, 40, 40])

current['very_low']  = fuzz.trimf(current.universe, [2, 2, 5])
current['low']       = fuzz.trimf(current.universe, [3, 6, 9])
current['medium']    = fuzz.trimf(current.universe, [7, 10, 13])
current['high']      = fuzz.trimf(current.universe, [11, 15, 18])
current['very_high'] = fuzz.trimf(current.universe, [16, 20, 20])

# ---------- Rule base (12+ rules) ----------
rules = [
    ctrl.Rule(soc['high'] & bt['hot'], current['very_low']),
    ctrl.Rule(soc['high'] & bt['warm'], current['low']),
    ctrl.Rule(soc['high'] & bt['normal'], current['low']),

    ctrl.Rule(soc['medium'] & bt['hot'], current['low']),
    ctrl.Rule(soc['medium'] & bt['warm'], current['medium']),
    ctrl.Rule(soc['medium'] & bt['normal'], current['high']),

    ctrl.Rule(soc['low'] & bt['hot'], current['low']),
    ctrl.Rule(soc['low'] & bt['warm'], current['high']),
    ctrl.Rule(soc['low'] & bt['normal'], current['very_high']),

    ctrl.Rule(dv['large'], current['very_low']),
    ctrl.Rule(dv['medium'], current['low']),
    ctrl.Rule(dv['small'], current['high']),

    ctrl.Rule(at['hot'], current['low']),
    ctrl.Rule(at['cool'] & soc['low'], current['very_high']),
    ctrl.Rule(at['normal'] & soc['low'], current['high']),
]

system = ctrl.ControlSystem(rules)


def level_label(I):
    if I >= 16: return "VERY HIGH"
    elif I >= 12: return "HIGH"
    elif I >= 8: return "MEDIUM"
    elif I >= 4: return "LOW"
    else: return "VERY LOW"


def get_charging_current(SOC, BT, DV, AT, verbose=False):
    """Run one simulation of the Mamdani FIS and return the recommended
    charging current in Amps (float)."""
    sim = ctrl.ControlSystemSimulation(system)
    sim.input['soc'] = SOC
    sim.input['bt']  = BT
    sim.input['dv']  = DV
    sim.input['at']  = AT
    sim.compute()
    I = sim.output['current']
    if verbose:
        print(f"SOC={SOC}%  BT={BT}C  DV={DV}mV  AT={AT}C  ->  I={I:.2f} A  [{level_label(I)}]")
    return I

print("Fuzzy Battery Charging Controller ready.")

## Phase 4 — Testing: 10 Different Input Combinations

Ten scenarios were chosen to exercise every corner of the input space: low/medium/high SOC,
normal/warm/hot battery temperature, small/medium/large cell-voltage imbalance, and cool/normal/hot
ambient temperature — including two deliberate stress tests (a fully "ideal" case and a "worst case").

In [ ]:
import pandas as pd

test_cases = [
    ("T1  Ideal safe fast charge",           5,  20, 5,  20),
    ("T2  Low SOC, cool ambient",            10, 18, 8,  16),
    ("T3  Low SOC, battery overheating",     12, 58, 10, 25),
    ("T4  Medium SOC, normal conditions",    50, 24, 10, 25),
    ("T5  Medium SOC, warm battery",         55, 38, 25, 28),
    ("T6  Medium SOC, large cell imbalance", 50, 26, 85, 25),
    ("T7  High SOC, normal battery temp",    85, 22, 8,  24),
    ("T8  High SOC, hot battery",            90, 58, 15, 30),
    ("T9  Low SOC, hot ambient day",         15, 24, 6,  38),
    ("T10 Worst-case multi-stress",          92, 59, 90, 39),
]

results = []
for name, SOC, BT, DV, AT in test_cases:
    I = get_charging_current(SOC, BT, DV, AT)
    results.append({
        "Scenario": name, "SOC (%)": SOC, "Batt Temp (C)": BT,
        "Cell dV (mV)": DV, "Ambient (C)": AT,
        "Charging Current (A)": round(I, 2), "Level": level_label(I)
    })

df = pd.DataFrame(results)
df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
colors = {"VERY HIGH":"#1a9850","HIGH":"#91cf60","MEDIUM":"#fee08b","LOW":"#fc8d59","VERY LOW":"#d73027"}
bar_colors = [colors[l] for l in df["Level"]]
ax.bar(df["Scenario"], df["Charging Current (A)"], color=bar_colors)
ax.set_ylabel("Recommended Charging Current (A)")
ax.set_title("Phase 4 — Charging Current for 10 Test Input Combinations")
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.savefig("phase4_test_results.png", dpi=150)
plt.show()

### Reference output (for comparison when you run this notebook)

| Scenario | SOC% | BT °C | dV mV | AT °C | Current (A) | Level |
|---|---:|---:|---:|---:|---:|---|
| T1  Ideal safe fast charge | 5 | 20 | 5 | 20 | 15.72 | HIGH |
| T2  Low SOC, cool ambient | 10 | 18 | 8 | 16 | 16.01 | VERY HIGH |
| T3  Low SOC, battery overheating | 12 | 58 | 10 | 25 | 10.65 | MEDIUM |
| T4  Medium SOC, normal conditions | 50 | 24 | 10 | 25 | 14.66 | HIGH |
| T5  Medium SOC, warm battery | 55 | 38 | 25 | 28 | 10.28 | MEDIUM |
| T6  Medium SOC, large cell imbalance | 50 | 26 | 85 | 25 | 11.47 | MEDIUM |
| T7  High SOC, normal battery temp | 85 | 22 | 8 | 24 | 10.82 | MEDIUM |
| T8  High SOC, hot battery | 90 | 58 | 15 | 30 | 10.50 | MEDIUM |
| T9  Low SOC, hot ambient day | 15 | 24 | 6 | 38 | 12.15 | HIGH |
| T10 Worst-case multi-stress | 92 | 59 | 90 | 39 | 5.09 | LOW |

*(Computed with an independent triangular-MF / min–max / centroid re-implementation of the same rule base, as a sanity check — skfuzzy should return values within ~0.1–0.3 A of these.)*

## Phase 5 — Analysis: Controller Behaviour, Strengths, Weaknesses, Scope of Improvement

### Observed behaviour
- **General trend:** current is highest when SOC is low, the battery is at normal temperature, the cell-voltage
  difference is small, and the ambient temperature is moderate (T1, T2, T4) — and lowest when several stress
  factors combine at once (T10: high SOC + hot battery + large imbalance + hot ambient → only 5.1 A).
- **Graceful degradation:** rather than an on/off cut-off, the recommended current tapers smoothly as SOC rises
  (T2 → T4 → T7) or as battery temperature rises (T1 → T5 → T8), which is exactly the "gradual, context-aware"
  behaviour threshold logic cannot provide.
- **Rule interaction / partial overrides:** the "hot battery" cases (T3, T8) are pulled *up* into the MEDIUM band
  instead of collapsing to LOW/VERY LOW, because the independent `dv['small'] → high` rule fires strongly
  (small cell-voltage imbalance) and is aggregated with equal weight (MAX) against the thermal-safety rules.
  This shows the rule base does not fully prioritise thermal safety over the other three inputs.

### Strengths
1. **Multi-input, human-interpretable reasoning** — 15 linguistic IF–THEN rules replace a rigid lookup table
   and are easy to explain/justify to a reviewer, unlike a black-box ML model.
2. **Smooth, graded output** — overlapping triangular membership functions avoid abrupt current jumps at
   threshold boundaries, reducing switching stress on the charger hardware.
3. **Fast, deterministic, lightweight** — no training data or iterative optimisation is needed; it runs in
   real time on modest hardware (a BMS microcontroller class device).
4. **Extensible rule structure** — new rules or inputs (e.g. cell temperature spread) can be added
   incrementally without redesigning the whole system.

### Weaknesses
1. **Independent-input rules can override safety-critical rules.** Because `dv` and `at` rules fire on their
   own (not always AND-ed with `soc`/`bt`) and MAX-aggregation treats every activated rule equally, a small
   voltage imbalance can partially cancel out a genuine over-temperature warning (see T3, T8 above).
2. **No temperature *rate-of-rise* input**, even though the project's own PPT lists it as a fifth input —
   the implemented code only reads instantaneous battery temperature, so a battery heating up rapidly but
   still numerically "warm" is treated the same as one that is stable at that temperature.
3. **Static membership functions** — the triangular breakpoints (e.g. `bt['hot'] = [45,60,60]`) are fixed by
   hand and were not tuned against real cell datasheets or thermal-runaway thresholds, so the boundaries are
   somewhat arbitrary.
4. **No hysteresis / rate limiting on the output** — because each call is stateless, the recommended current
   could in principle oscillate between calls if a sensor reading jitters near a rule boundary, which is
   undesirable for physical charger hardware.
5. **Manual rule-weight tuning** — with 15 hand-written rules there is no guarantee the rule base is complete
   or non-conflicting for every corner of the 4‑D input space; some combinations are covered by only one
   weak rule (e.g. `bt['warm']` is never directly combined with `dv` or `at`).

### Scope of improvement
- **Add the missing 5th input** (temperature rise rate, °C/min) as shown in the PPT, and always AND it with
  `bt` so a rapidly-heating "warm" battery is treated more conservatively than a stable one.
- **Re-weight or restructure the rule base** so thermal-safety rules (`bt['hot']`, fast rise rate) always take
  priority over `dv`/`at` rules — e.g. by using a hierarchical/cascaded FIS (first decide a safety ceiling from
  SOC+BT, then let dv/at only reduce, never increase, that ceiling).
- **Tune membership-function breakpoints** using real cell datasheet limits (e.g. manufacturer's max
  charge temperature) or a genetic algorithm (this course pairs Fuzzy Logic with Genetic Programming) to
  optimise the MF parameters against a labelled dataset of "ideal" charging decisions.
- **Add output smoothing / rate limiting** in the surrounding control loop so the recommended current cannot
  jump by more than a fixed step between consecutive BMS control cycles.
- **Validate against real battery test data** (charge/discharge cycler logs) instead of only synthetic test
  inputs, and compare cycle life / temperature rise against the existing threshold controller.

## Phase 6 — User Interface

An interactive UI built with `ipywidgets`: sliders for the four inputs, a **Compute** button, and a live
readout of the recommended charging current with its safety level and a small gauge-style bar.

> If `ipywidgets` is not installed, run `pip install ipywidgets` (and `jupyter nbextension enable --py widgetsnbextension` for classic Notebook; not needed for JupyterLab ≥3 or Google Colab).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

style = {'description_width': '160px'}
layout = widgets.Layout(width='480px')

soc_slider = widgets.FloatSlider(value=50, min=0, max=100, step=1,
                description='State of Charge (%)', style=style, layout=layout)
bt_slider  = widgets.FloatSlider(value=25, min=0, max=60, step=1,
                description='Battery Temp (C)', style=style, layout=layout)
dv_slider  = widgets.FloatSlider(value=10, min=0, max=100, step=1,
                description='Cell Voltage Diff (mV)', style=style, layout=layout)
at_slider  = widgets.FloatSlider(value=25, min=15, max=40, step=1,
                description='Ambient Temp (C)', style=style, layout=layout)

compute_btn = widgets.Button(description='Compute Charging Current',
                button_style='success', icon='bolt', layout=widgets.Layout(width='250px'))
output_box = widgets.Output()

LEVEL_COLOR = {"VERY HIGH":"#1a9850","HIGH":"#91cf60","MEDIUM":"#fee08b",
               "LOW":"#fc8d59","VERY LOW":"#d73027"}

def on_click(b):
    with output_box:
        clear_output()
        I = get_charging_current(soc_slider.value, bt_slider.value,
                                  dv_slider.value, at_slider.value)
        lvl = level_label(I)
        pct = max(0, min(100, (I - 2) / (20 - 2) * 100))
        html = f"""
        <div style='font-family:sans-serif;padding:14px;border-radius:10px;
                    background:#f4f4f8;border:1px solid #ccc;width:480px'>
          <div style='font-size:15px;color:#555'>Recommended Charging Current</div>
          <div style='font-size:36px;font-weight:700;color:{LEVEL_COLOR[lvl]}'>{I:.2f} A</div>
          <div style='font-size:14px;font-weight:600;color:{LEVEL_COLOR[lvl]};margin-bottom:8px'>{lvl}</div>
          <div style='background:#e0e0e0;border-radius:6px;height:14px;width:100%'>
            <div style='background:{LEVEL_COLOR[lvl]};height:14px;border-radius:6px;width:{pct:.0f}%'></div>
          </div>
        </div>
        """
        display(widgets.HTML(html))

compute_btn.on_click(on_click)

ui = widgets.VBox([
    widgets.HTML("<h3>EV Battery Charging Controller — Live Simulator</h3>"),
    soc_slider, bt_slider, dv_slider, at_slider,
    compute_btn, output_box
])
display(ui)
on_click(None)  # show an initial result

**How to use / screenshot for submission:** move the four sliders to any combination you like
(or re-run the 10 Phase‑4 scenarios above by typing the values in) and click **Compute Charging Current** —
the card below updates instantly with the recommended current, its safety level, and a colour-coded gauge bar.
Take a screenshot of this cell's output for the assignment submission.